# Exploratory Data Analysis and Preprocessing

This notebook performs exploratory data analysis (EDA) and preprocessing on the IMDB movie review dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from wordcloud import WordCloud
from collections import Counter

# Add src to path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

# Import our modules
from data_loader import DataLoader
from preprocessing import TextPreprocessor
from utils import setup_logging, validate_data_structure, calculate_text_statistics

# Setup
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Setup logging
setup_logging("INFO")

print("Libraries imported successfully!")

## 1. Load and Inspect Data

In [ ]:
# Load the dataset
data_path = "../data/IMDB Dataset.csv"
loader = DataLoader(data_path)

# Load data
df = loader.load_data()

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Check data quality
quality_report = loader.check_data_quality()
print("Data Quality Report:")
for key, value in quality_report.items():
    print(f"{key}: {value}")

In [ ]:
# Validate data structure
try:
    validate_data_structure(df)
    print("✓ Data structure validation passed")
except ValueError as e:
    print(f"✗ Data structure validation failed: {e}")

## 2. Data Cleaning

In [ ]:
# Clean the data
df_clean = loader.clean_data()

print(f"Original dataset size: {len(df)}")
print(f"Cleaned dataset size: {len(df_clean)}")
print(f"Removed {len(df) - len(df_clean)} rows")

# Check for any remaining issues
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

print("\nData types:")
print(df_clean.dtypes)

## 3. Exploratory Data Analysis

### 3.1 Class Distribution

In [ ]:
# Analyze sentiment distribution
sentiment_counts = df_clean['sentiment'].value_counts()
print("Sentiment Distribution:")
print(sentiment_counts)
print(f"\nPositive: {sentiment_counts['positive']} ({sentiment_counts['positive']/len(df_clean)*100:.1f}%)")
print(f"Negative: {sentiment_counts['negative']} ({sentiment_counts['negative']/len(df_clean)*100:.1f}%)")

# Create visualization
plt.figure(figsize=(10, 5))

# Bar plot
plt.subplot(1, 2, 1)
sns.countplot(data=df_clean, x='sentiment')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')

# Pie chart
plt.subplot(1, 2, 2)
plt.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Sentiment Percentage')

plt.tight_layout()
plt.show()

### 3.2 Text Length Analysis

In [ ]:
# Calculate text statistics
df_clean['word_count'] = df_clean['review'].apply(lambda x: len(str(x).split()))
df_clean['char_count'] = df_clean['review'].apply(lambda x: len(str(x)))
df_clean['avg_word_length'] = df_clean['char_count'] / df_clean['word_count']

print("Text Statistics:")
print(df_clean[['word_count', 'char_count', 'avg_word_length']].describe())

# Visualize text length distribution
plt.figure(figsize=(15, 10))

# Word count distribution
plt.subplot(2, 2, 1)
sns.histplot(data=df_clean, x='word_count', bins=50, kde=True)
plt.title('Word Count Distribution')
plt.xlabel('Word Count')
plt.ylabel('Frequency')

# Word count by sentiment
plt.subplot(2, 2, 2)
sns.boxplot(data=df_clean, x='sentiment', y='word_count')
plt.title('Word Count by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Word Count')

# Character count distribution
plt.subplot(2, 2, 3)
sns.histplot(data=df_clean, x='char_count', bins=50, kde=True)
plt.title('Character Count Distribution')
plt.xlabel('Character Count')
plt.ylabel('Frequency')

# Character count by sentiment
plt.subplot(2, 2, 4)
sns.boxplot(data=df_clean, x='sentiment', y='char_count')
plt.title('Character Count by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Character Count')

plt.tight_layout()
plt.show()

### 3.3 Sample Reviews Inspection

In [ ]:
# Display sample reviews for each sentiment
print("=== SAMPLE POSITIVE REVIEWS ===")
positive_reviews = df_clean[df_clean['sentiment'] == 'positive']['review'].head(3)
for i, review in enumerate(positive_reviews, 1):
    print(f"\nReview {i}:")
    print(f"Length: {len(review.split())} words")
    print(f"Text: {review[:300]}...")

print("\n" + "="*50)
print("=== SAMPLE NEGATIVE REVIEWS ===")
negative_reviews = df_clean[df_clean['sentiment'] == 'negative']['review'].head(3)
for i, review in enumerate(negative_reviews, 1):
    print(f"\nReview {i}:")
    print(f"Length: {len(review.split())} words")
    print(f"Text: {review[:300]}...")

### 3.4 Word Frequency Analysis

In [ ]:
# Word frequency analysis (before preprocessing)
def get_word_frequency(texts, top_n=20):
    all_words = []
    for text in texts:
        words = str(text).lower().split()
        all_words.extend(words)
    
    word_freq = Counter(all_words)
    return word_freq.most_common(top_n)

# Get word frequencies for each sentiment
positive_words = get_word_frequency(df_clean[df_clean['sentiment'] == 'positive']['review'])
negative_words = get_word_frequency(df_clean[df_clean['sentiment'] == 'negative']['review'])

# Create visualization
plt.figure(figsize=(15, 6))

# Positive words
plt.subplot(1, 2, 1)
pos_words, pos_counts = zip(*positive_words[:15])
plt.barh(range(len(pos_words)), pos_counts)
plt.yticks(range(len(pos_words)), pos_words)
plt.title('Top 15 Words in Positive Reviews')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()

# Negative words
plt.subplot(1, 2, 2)
neg_words, neg_counts = zip(*negative_words[:15])
plt.barh(range(len(neg_words)), neg_counts)
plt.yticks(range(len(neg_words)), neg_words)
plt.title('Top 15 Words in Negative Reviews')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

print("Top 10 words in positive reviews:")
for word, count in positive_words[:10]:
    print(f"{word}: {count}")

print("\nTop 10 words in negative reviews:")
for word, count in negative_words[:10]:
    print(f"{word}: {count}")

### 3.5 Word Cloud Visualization

In [ ]:
# Generate word clouds
def generate_wordcloud(text, title):
    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          max_words=100, colormap='viridis').generate(text)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.show()

# Generate word clouds for each sentiment
positive_text = ' '.join(df_clean[df_clean['sentiment'] == 'positive']['review'])
negative_text = ' '.join(df_clean[df_clean['sentiment'] == 'negative']['review'])

generate_wordcloud(positive_text, 'Word Cloud - Positive Reviews')
generate_wordcloud(negative_text, 'Word Cloud - Negative Reviews')

## 4. Text Preprocessing

In [ ]:
# Initialize the preprocessor
preprocessor = TextPreprocessor(remove_stopwords=True, use_lemmatization=True)

# Test preprocessing on sample texts
sample_texts = [
    "This movie was absolutely fantastic! The acting was great and the plot was amazing.",
    "I hated this film. It was boring and predictable. <br/> Visit http://example.com for more info.",
    "The movie wasn't bad, but it could have been better. Some scenes were too long."
]

print("=== PREPROCESSING EXAMPLES ===")
for i, text in enumerate(sample_texts, 1):
    print(f"\nOriginal Text {i}:")
    print(text)
    print(f"\nProcessed Text {i}:")
    processed = preprocessor.preprocess_text(text)
    print(processed)
    print("-" * 80)

In [ ]:
# Preprocess a sample of the dataset for analysis
sample_size = 1000
sample_df = df_clean.sample(n=sample_size, random_state=42)

print(f"Preprocessing {sample_size} sample reviews...")
sample_df['processed_review'] = preprocessor.preprocess_batch(sample_df['review'].tolist())

# Analyze the effect of preprocessing
sample_df['processed_word_count'] = sample_df['processed_review'].apply(lambda x: len(x.split()))

print("\nEffect of preprocessing on text length:")
print(f"Original average word count: {sample_df['word_count'].mean():.2f}")
print(f"Processed average word count: {sample_df['processed_word_count'].mean():.2f}")
print(f"Average reduction: {(1 - sample_df['processed_word_count'].mean()/sample_df['word_count'].mean())*100:.1f}%")

# Visualize the effect
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(sample_df['word_count'], bins=30, alpha=0.7, label='Original', color='blue')
plt.hist(sample_df['processed_word_count'], bins=30, alpha=0.7, label='Processed', color='orange')
plt.title('Word Count Distribution Before/After Preprocessing')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
sns.scatterplot(data=sample_df, x='word_count', y='processed_word_count', alpha=0.6)
plt.plot([0, sample_df['word_count'].max()], [0, sample_df['word_count'].max()], 'r--', alpha=0.8)
plt.title('Original vs Processed Word Count')
plt.xlabel('Original Word Count')
plt.ylabel('Processed Word Count')

plt.tight_layout()
plt.show()

### 4.1 Word Frequency After Preprocessing

In [ ]:
# Analyze word frequencies after preprocessing
positive_processed = get_word_frequency(
    sample_df[sample_df['sentiment'] == 'positive']['processed_review']
)
negative_processed = get_word_frequency(
    sample_df[sample_df['sentiment'] == 'negative']['processed_review']
)

# Create visualization
plt.figure(figsize=(15, 6))

# Positive words after preprocessing
plt.subplot(1, 2, 1)
pos_words, pos_counts = zip(*positive_processed[:15])
plt.barh(range(len(pos_words)), pos_counts)
plt.yticks(range(len(pos_words)), pos_words)
plt.title('Top 15 Words in Positive Reviews (After Preprocessing)')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()

# Negative words after preprocessing
plt.subplot(1, 2, 2)
neg_words, neg_counts = zip(*negative_processed[:15])
plt.barh(range(len(neg_words)), neg_counts)
plt.yticks(range(len(neg_words)), neg_words)
plt.title('Top 15 Words in Negative Reviews (After Preprocessing)')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

print("Top 10 words in positive reviews (after preprocessing):")
for word, count in positive_processed[:10]:
    print(f"{word}: {count}")

print("\nTop 10 words in negative reviews (after preprocessing):")
for word, count in negative_processed[:10]:
    print(f"{word}: {count}")

## 5. Data Splitting

In [ ]:
# Split the data into train, validation, and test sets
train_df, val_df, test_df = loader.split_data(test_size=0.2, val_size=0.1)

print("Data Split Summary:")
print(f"Training set: {len(train_df)} samples ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"Validation set: {len(val_df)} samples ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"Test set: {len(test_df)} samples ({len(test_df)/len(df_clean)*100:.1f}%)")

# Check sentiment distribution in each split
print("\nSentiment distribution in splits:")
for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    sentiment_dist = split_df['sentiment'].value_counts(normalize=True)
    print(f"{split_name}: Positive {sentiment_dist['positive']:.2%}, Negative {sentiment_dist['negative']:.2%}")

# Save the splits
loader.save_splits(train_df, val_df, test_df)
print("\nData splits saved to data/ directory")

## 6. Summary and Insights

In [ ]:
# Create a summary of findings
summary = {
    'dataset_info': {
        'total_samples': len(df_clean),
        'positive_samples': len(df_clean[df_clean['sentiment'] == 'positive']),
        'negative_samples': len(df_clean[df_clean['sentiment'] == 'negative']),
        'balance_ratio': len(df_clean[df_clean['sentiment'] == 'positive']) / len(df_clean[df_clean['sentiment'] == 'negative'])
    },
    'text_statistics': {
        'avg_word_count': df_clean['word_count'].mean(),
        'median_word_count': df_clean['word_count'].median(),
        'min_word_count': df_clean['word_count'].min(),
        'max_word_count': df_clean['word_count'].max(),
        'std_word_count': df_clean['word_count'].std()
    },
    'preprocessing_impact': {
        'avg_word_reduction_percent': (1 - sample_df['processed_word_count'].mean()/sample_df['word_count'].mean())*100,
        'common_positive_words': [word for word, count in positive_processed[:5]],
        'common_negative_words': [word for word, count in negative_processed[:5]]
    },
    'data_splits': {
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df)
    }
}

print("=== DATASET ANALYSIS SUMMARY ===")
print(f"\nDataset Info:")
for key, value in summary['dataset_info'].items():
    print(f"  {key}: {value}")

print(f"\nText Statistics:")
for key, value in summary['text_statistics'].items():
    print(f"  {key}: {value:.2f}")

print(f"\nPreprocessing Impact:")
for key, value in summary['preprocessing_impact'].items():
    print(f"  {key}: {value}")

print(f"\nData Splits:")
for key, value in summary['data_splits'].items():
    print(f"  {key}: {value}")

print("\n=== KEY INSIGHTS ===")
print("1. Dataset is balanced with equal positive and negative reviews")
print("2. Reviews vary significantly in length (need to consider this in modeling)")
print("3. Preprocessing reduces text length by ~40-60% while preserving meaning")
print("4. After preprocessing, sentiment-specific words become more prominent")
print("5. Data is properly stratified to maintain balance across splits")